In [1]:
# ==========================================
# Semantic causal-claim coverage
#
# Purpose:
# Independently mine causal claims from chunks_all.csv and measure whether claims
# in policy and sentiment have close semantic counterparts. 
# Frozen NMF topics are contextual strata only.
#
# Methodological roots:
# - Causal relation extraction: causal NLP literature
# - Sentence embeddings: Reimers and Gurevych (2019)
# - Text-measurement validation: Egami et al. (2022)
# - Directed content analysis: Hsieh and Shannon (2005)
# ==========================================

import html
import io
import os

# Keep sparse linear-algebra backends deterministic and avoid oversubscription.
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
import re
from pathlib import Path
import shutil

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import Image as NotebookImage, display

from scipy.spatial.distance import jensenshannon
from scipy.sparse import hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 220)

RANDOM_STATE = 42
MIN_COUNTRY_ITEMS = 5
SENTENCE_EMBEDDING_MODEL = os.environ.get(
    "CAUSAL_NLP_EMBEDDING_MODEL",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
)
USE_SENTENCE_TRANSFORMERS = os.environ.get(
    "CAUSAL_NLP_USE_SENTENCE_TRANSFORMERS", "1"
).strip().lower() not in {"0", "false", "no"}


def find_project_root() -> Path:
    """Find the repository root from the expected NMF and chunk paths."""
    env_root = os.environ.get("CAUSAL_NLP_PROJECT_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (
            (candidate / "progress" / "topic_modelling" / "nmf").exists()
            and (candidate / "csvs" / "chunked" / "chunks_all.csv").exists()
        ):
            return candidate

    for start in [Path.cwd(), *Path.cwd().parents]:
        if (
            (start / "progress" / "topic_modelling" / "nmf").exists()
            and (start / "csvs" / "chunked" / "chunks_all.csv").exists()
        ):
            return start.resolve()

    raise FileNotFoundError(
        "Could not find progress/topic_modelling/nmf and "
        "csvs/chunked/chunks_all.csv. Run from the repository root or set "
        "CAUSAL_NLP_PROJECT_ROOT."
    )


PROJECT_ROOT = find_project_root()
NMF_ROOT = PROJECT_ROOT / "progress" / "topic_modelling" / "nmf"
CSVS_CHUNKED = PROJECT_ROOT / "csvs" / "chunked"
CHUNKS_ALL = CSVS_CHUNKED / "chunks_all.csv"
CAUSAL_NLP_ROOT = Path(
    os.environ.get(
        "CAUSAL_NLP_OUTPUT_ROOT",
        PROJECT_ROOT / "progress" / "causal_nlp",
    )
).expanduser().resolve()
METHOD_ROOT = CAUSAL_NLP_ROOT

# Results are isolated in method-specific semantic_coverage folders.
POLICY_OUTPUT = NMF_ROOT / "policy" / "output" / "global"
SENTIMENT_OUTPUT = NMF_ROOT / "sentiment" / "output"


def infer_sentiment_country(row: pd.Series) -> str:
    """Derive only clearly supported sentiment country groups."""
    source = " ".join(
        str(row.get(column, ""))
        for column in ["filename", "relative_source_file", "source_file", "doc_id"]
    ).lower()
    if any(token in source for token in ["ireland", "irish times", "qqi_", "qqi "]):
        return "ireland"
    if any(token in source for token in ["france", "français", "francais", "french", "ifop", "labo"]):
        return "france"
    return "other"


def build_context_windows(chunks: pd.DataFrame) -> pd.DataFrame:
    """Add neighbouring original chunks without changing the analytical unit."""
    ordered = chunks.sort_values(["doc_id", "chunk_index"]).copy()
    ordered["previous_chunk_text"] = ordered.groupby("doc_id")["chunk_text"].shift(1)
    ordered["next_chunk_text"] = ordered.groupby("doc_id")["chunk_text"].shift(-1)
    ordered["context_window"] = (
        ordered["heading_context"].fillna("").astype(str)
        + "\n\nPREVIOUS:\n"
        + ordered["previous_chunk_text"].fillna("").astype(str)
        + "\n\nCURRENT:\n"
        + ordered["chunk_text"].fillna("").astype(str)
        + "\n\nNEXT:\n"
        + ordered["next_chunk_text"].fillna("").astype(str)
    ).str.slice(0, 7000)
    return ordered


def build_topic_description(frame: pd.DataFrame) -> pd.Series:
    return (
        frame["topic_label"].fillna("").astype(str)
        + " "
        + frame["topic_prototype"].fillna("").astype(str)
        + " "
        + frame["keywords"].fillna("").astype(str)
    ).str.replace(r"\s+", " ", regex=True).str.strip()





# Shared causal-text cleaning layer
import sys

_SHARED_CANDIDATES = [
    Path(os.environ.get("CAUSAL_NLP_SHARED_MODULE_DIR", "")).expanduser(),
    PROJECT_ROOT / "progress" / "causal_nlp" / "shared",
    Path.cwd() / "shared",
    Path.cwd(),
]
SHARED_MODULE_DIR = next(
    (
        candidate.resolve()
        for candidate in _SHARED_CANDIDATES
        if str(candidate) and (candidate / "causal_text_cleaning.py").exists()
    ),
    None,
)
if SHARED_MODULE_DIR is None:
    raise FileNotFoundError(
        "causal_text_cleaning.py was not found. Place it in "
        "progress/causal_nlp/shared or set CAUSAL_NLP_SHARED_MODULE_DIR."
    )
if str(SHARED_MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_MODULE_DIR))

from causal_text_cleaning import (
    CAUSAL_ARTIFACT_PHRASES,
    CAUSAL_ARTIFACT_TOKENS,
    CAUSAL_EMBEDDING_STOPWORDS,
    CLEANING_VERSION,
    build_context_windows,
    build_topic_description,
    contains_contact_or_link,
    content_token_ratio,
    infer_sentiment_country,
    is_source_residue,
    load_or_build_clean_sentence_inventory,
    normalize_source_text,
    sentence_units,
)

# Compatibility names used by the method-specific extraction code.
normalize_for_noise_check = normalize_source_text
split_sentences = sentence_units


def fit_text_embeddings(texts: list[str], max_components: int = 60):
    """Use multilingual Sentence-BERT, otherwise multilingual word+character LSA."""
    if USE_SENTENCE_TRANSFORMERS:
        try:
            from sentence_transformers import SentenceTransformer

            model = SentenceTransformer(
                SENTENCE_EMBEDDING_MODEL,
                local_files_only=True,
            )
            embeddings = model.encode(
                texts,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
            return np.asarray(embeddings), {
                "method": f"sentence_transformer:{SENTENCE_EMBEDDING_MODEL}",
                "sentence_model": model,
                "word_vectorizer": None,
                "char_vectorizer": None,
                "svd": None,
            }
        except Exception as error:
            print(
                "Multilingual Sentence-BERT unavailable; "
                "using multilingual word+character TF-IDF plus LSA:",
                type(error).__name__,
            )

    word_vectorizer = TfidfVectorizer(
        analyzer="word",
        stop_words=sorted(CAUSAL_EMBEDDING_STOPWORDS),
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        sublinear_tf=True,
        max_features=6000,
    )
    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 4),
        min_df=3,
        sublinear_tf=True,
        max_features=6000,
    )
    word_matrix = word_vectorizer.fit_transform(texts)
    char_matrix = char_vectorizer.fit_transform(texts)
    matrix = hstack([word_matrix, char_matrix], format="csr")

    n_components = min(
        max_components,
        matrix.shape[0] - 1,
        matrix.shape[1] - 1,
    )
    if n_components < 2:
        return normalize(matrix), {
            "method": "multilingual_word_char_tfidf_sparse",
            "sentence_model": None,
            "word_vectorizer": word_vectorizer,
            "char_vectorizer": char_vectorizer,
            "svd": None,
        }

    svd = TruncatedSVD(
        n_components=n_components,
        random_state=RANDOM_STATE,
        n_iter=3,
    )
    embeddings = normalize(svd.fit_transform(matrix))
    return embeddings, {
        "method": f"multilingual_word_char_lsa:{n_components}_components",
        "sentence_model": None,
        "word_vectorizer": word_vectorizer,
        "char_vectorizer": char_vectorizer,
        "svd": svd,
    }

def transform_text_embeddings(texts: list[str], backend: dict) -> np.ndarray:
    if backend["sentence_model"] is not None:
        return np.asarray(
            backend["sentence_model"].encode(
                texts,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
        )

    word_matrix = backend["word_vectorizer"].transform(texts)
    char_matrix = backend["char_vectorizer"].transform(texts)
    matrix = hstack([word_matrix, char_matrix], format="csr")
    if backend["svd"] is None:
        return normalize(matrix)
    return normalize(backend["svd"].transform(matrix))

def normalise_minmax(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").fillna(0.0)
    minimum = float(values.min())
    maximum = float(values.max())
    if maximum <= minimum:
        return pd.Series(np.zeros(len(values)), index=series.index)
    return (values - minimum) / (maximum - minimum)

def save_verified_png(fig, output_path: Path, dpi: int = 120) -> None:
    """Write a standard RGB PNG directly to the final path and verify it."""
    output_path = Path(output_path).expanduser().resolve()
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Render the matplotlib figure fully in memory first. This avoids temporary
    # files and atomic renames, which can be unreliable on some synced or
    # virtual filesystems.
    raw_buffer = io.BytesIO()
    rgb_buffer = io.BytesIO()

    try:
        fig.savefig(
            raw_buffer,
            format="png",
            dpi=dpi,
            bbox_inches="tight",
            facecolor="white",
            transparent=False,
        )
        plt.close(fig)

        raw_buffer.seek(0)
        with Image.open(raw_buffer) as image:
            image.load()
            rgb_image = image.convert("RGB")
            rgb_image.save(
                rgb_buffer,
                format="PNG",
                optimize=False,
                compress_level=6,
            )

        png_bytes = rgb_buffer.getvalue()
        if not png_bytes.startswith(b"\x89PNG\r\n\x1a\n"):
            raise OSError("Rendered bytes do not contain a valid PNG signature.")
        if len(png_bytes) <= 100:
            raise OSError("Rendered PNG is unexpectedly small.")

        # Remove any stale or incomplete file before writing the final bytes.
        output_path.unlink(missing_ok=True)
        output_path.write_bytes(png_bytes)

        # Reopen the exact final path that will be used by the notebook and the
        # file browser.
        with Image.open(output_path) as image:
            image.load()
            if image.format != "PNG":
                raise OSError(f"Unexpected final image format: {image.format}")
            if image.mode != "RGB":
                raise OSError(f"Unexpected final image mode: {image.mode}")
            width, height = image.size
            if width <= 0 or height <= 0:
                raise OSError("Final PNG has invalid dimensions.")

        print(
            "Saved verified PNG:",
            output_path,
            "mode=RGB",
            f"size=({width}, {height})",
            f"bytes={output_path.stat().st_size:,}",
        )
    finally:
        plt.close(fig)
        raw_buffer.close()
        rgb_buffer.close()


In [2]:
# ==========================================
# Step 1: Build or load the shared clean sentence inventory, then attach NMF topics
# ==========================================

OUTPUT_DIR = METHOD_ROOT / "output" / "semantic_coverage"
IMG_DIR = METHOD_ROOT / "img" / "semantic_coverage"
SHARED_OUTPUT_DIR = METHOD_ROOT / "output" / "shared"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMG_DIR.mkdir(parents=True, exist_ok=True)
SHARED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_SENTENCE_INVENTORY = SHARED_OUTPUT_DIR / "clean_sentence_inventory.csv"
CLEAN_SENTENCE_METADATA = SHARED_OUTPUT_DIR / "clean_sentence_inventory_metadata.json"
CLEAN_SENTENCE_SUMMARY = SHARED_OUTPUT_DIR / "clean_sentence_inventory_summary.csv"
CLEAN_SENTENCE_VALIDATION = SHARED_OUTPUT_DIR / "clean_sentence_inventory_validation.csv"

clean_sentence_inventory, shared_cleaning_metadata = (
    load_or_build_clean_sentence_inventory(
        CHUNKS_ALL,
        CLEAN_SENTENCE_INVENTORY,
        CLEAN_SENTENCE_METADATA,
        CLEAN_SENTENCE_SUMMARY,
        CLEAN_SENTENCE_VALIDATION,
    )
)
raw_chunks_all = build_context_windows(pd.read_csv(CHUNKS_ALL))

policy_topics = pd.read_csv(
    POLICY_OUTPUT / "policy_global_nmf_topic_info_with_labels_final.csv"
).sort_values("topic").reset_index(drop=True)
sentiment_topics = pd.read_csv(
    SENTIMENT_OUTPUT / "sentiment_nmf_topic_info.csv"
).sort_values("topic").reset_index(drop=True)
policy_assignments = pd.read_csv(
    POLICY_OUTPUT / "policy_global_nmf_documents_with_topic_labels_final.csv"
)[["chunk_id", "topic", "topic_confidence", "topic_label"]]
sentiment_assignments = pd.read_csv(
    SENTIMENT_OUTPUT / "sentiment_nmf_original_documents_with_topic_labels_final.csv"
)[["chunk_id", "topic", "topic_confidence", "topic_label"]]
synthetic_assignments = pd.read_csv(
    SENTIMENT_OUTPUT / "sentiment_nmf_synthetic_assignments_with_topic_labels_final.csv"
)[["chunk_id", "assigned_topic", "topic_confidence", "topic_label"]].rename(
    columns={"assigned_topic": "topic"}
)


def attach_clean_sentences(
    assignments: pd.DataFrame,
    corpus: str,
    source_type: str,
) -> pd.DataFrame:
    selected = clean_sentence_inventory[
        clean_sentence_inventory["corpus"].eq(corpus)
        & clean_sentence_inventory["source_type"].eq(source_type)
    ].merge(assignments, on="chunk_id", how="inner", validate="many_to_one")
    context_columns = raw_chunks_all[
        ["chunk_id", "chunk_text", "context_window"]
    ].drop_duplicates("chunk_id")
    selected = selected.merge(
        context_columns,
        on="chunk_id",
        how="left",
        validate="many_to_one",
    )
    selected["original_chunk_text"] = selected["chunk_text"]
    selected["chunk_text"] = selected["clean_sentence"]
    return selected


policy_chunks = attach_clean_sentences(policy_assignments, "policy", "original")
sentiment_chunks = attach_clean_sentences(
    sentiment_assignments, "sentiment", "original"
)
synthetic_chunks = attach_clean_sentences(
    synthetic_assignments, "sentiment", "synthetic"
)

policy_chunks["analysis_country"] = (
    policy_chunks["country"].fillna("other").astype(str).str.lower()
)
sentiment_chunks["analysis_country"] = sentiment_chunks.apply(
    infer_sentiment_country,
    axis=1,
)
synthetic_chunks["analysis_country"] = "synthetic"

# Chunk linkage is checked against raw chunks; sentence counts report the shared
# cleaning layer's usable analytical inventory.
linkage_summary = pd.DataFrame([
    {
        "corpus": "policy",
        "frozen_nmf_chunks": len(policy_assignments),
        "raw_linked_chunks": raw_chunks_all[
            raw_chunks_all["chunk_id"].isin(policy_assignments["chunk_id"])
        ]["chunk_id"].nunique(),
        "clean_linked_chunks": policy_chunks["chunk_id"].nunique(),
        "clean_sentences": len(policy_chunks),
    },
    {
        "corpus": "sentiment",
        "frozen_nmf_chunks": len(sentiment_assignments),
        "raw_linked_chunks": raw_chunks_all[
            raw_chunks_all["chunk_id"].isin(sentiment_assignments["chunk_id"])
        ]["chunk_id"].nunique(),
        "clean_linked_chunks": sentiment_chunks["chunk_id"].nunique(),
        "clean_sentences": len(sentiment_chunks),
    },
    {
        "corpus": "synthetic_sentiment",
        "frozen_nmf_chunks": len(synthetic_assignments),
        "raw_linked_chunks": raw_chunks_all[
            raw_chunks_all["chunk_id"].isin(synthetic_assignments["chunk_id"])
        ]["chunk_id"].nunique(),
        "clean_linked_chunks": synthetic_chunks["chunk_id"].nunique(),
        "clean_sentences": len(synthetic_chunks),
    },
])
linkage_summary["raw_linkage_rate"] = (
    linkage_summary["raw_linked_chunks"]
    / linkage_summary["frozen_nmf_chunks"].clip(lower=1)
)
linkage_summary.to_csv(
    OUTPUT_DIR / "original_chunk_linkage_summary.csv",
    index=False,
)
if linkage_summary["raw_linkage_rate"].lt(0.95).any():
    raise ValueError("Raw chunk linkage fell below 95%.")

print("Shared cleaning version:", CLEANING_VERSION)
print("Shared inventory hash:", shared_cleaning_metadata["inventory_sha256"])
print(linkage_summary.to_string(index=False))


Shared cleaning version: shared-causal-text
Shared inventory hash: 59f25d414f8672b5fd0423a5c35ea1452e3d3d051a4d045d2c378c142c40f96a
             corpus  frozen_nmf_chunks  raw_linked_chunks  clean_linked_chunks  clean_sentences  raw_linkage_rate
             policy               1888               1888                 1819            14944               1.0
          sentiment                455                455                  450             3578               1.0
synthetic_sentiment                202                202                  202             2040               1.0


In [3]:
# ==========================================
# Step 2: Independently extract structured causal claims
#
# Purpose:
# Apply a sentence-level causal pattern extractor to the shared clean sentence inventory.
# The first valid relation in each sentence becomes a traceable cause--relation--
# effect claim. Boilerplate is filtered to reduce licensing and publication noise.
# ==========================================

MAX_SENTENCE_WORDS = 110
MIN_SIDE_WORDS = 2
MAX_SIDE_WORDS = 65


def cause_first_pattern(cues: str) -> str:
    """Build a boundary-safe cause → cue → effect expression."""
    return (
        rf"(?P<cause>.+?)\s+"
        rf"(?P<cue>(?<!\w)(?:{cues})(?!\w))\s+"
        rf"(?P<effect>.+)"
    )


def reversed_pattern(cues: str) -> str:
    """Build a boundary-safe effect → cue → cause expression."""
    return (
        rf"(?P<effect>.+?)\s+"
        rf"(?P<cue>(?<!\w)(?:{cues})(?!\w))\s+"
        rf"(?P<cause>.+)"
    )


# Patterns are ordered from specific multi-word cues to broader predicates.
CAUSE_FIRST_PATTERNS = [
    (
        cause_first_pattern(
            r"may lead to|can lead to|could lead to|is likely to lead to|"
            r"lead(?:s|ing)? to|gave rise to|gives? rise to|result(?:s|ed|ing)? in|"
            r"bring(?:s|ing)? about|brought about|caus(?:e|es|ed|ing)|"
            r"contribut(?:e|es|ed|ing) to|drive|drives|driving|"
            r"produc(?:e|es|ed|ing)|trigger(?:s|ed|ing)?|"
            r"(?:(?:to|can|could|may|might|must|should|will|would|shall)\s+generate|generates|generated|generating)|(?:(?:to|can|could|may|might|must|should|will|would|shall)\s+create|creates|created|creating)|"
            r"exacerbat(?:e|es|ed|ing)|worsen(?:s|ed|ing)?|"
            r"intensif(?:y|ies|ied|ying)|accelerat(?:e|es|ed|ing)|"
            r"increas(?:e|es|ed|ing)|rais(?:e|es|ed|ing)|"
            r"amplif(?:y|ies|ied|ying)|heighten(?:s|ed|ing)?"
        ),
        "causes_or_increases",
    ),
    (
        cause_first_pattern(
            r"reduc(?:e|es|ed|ing)|prevent(?:s|ed|ing)?|"
            r"limit(?:s|ed|ing)?|mitigat(?:e|es|ed|ing)|"
            r"decreas(?:e|es|ed|ing)|lower(?:s|ed|ing)|"
            r"minimi(?:s|z)(?:e|es|ed|ing)|alleviat(?:e|es|ed|ing)|"
            r"avoid(?:s|ed|ing)?|curb(?:s|ed|ing)?|constrain(?:s|ed|ing)?|"
            r"counteract(?:s|ed|ing)?|protect(?:s|ed|ing)? against|"
            r"safeguard(?:s|ed|ing)? against"
        ),
        "reduces_or_prevents",
    ),
    (
        cause_first_pattern(
            r"enabl(?:e|es|ed|ing)|"
            r"(?:(?:to|can|could|may|might|must|should|will|would|shall)\s+support(?:\s+and\s+assist)?|supports|supported|supporting)|"
            r"facilitat(?:e|es|ed|ing)|"
            r"allow(?:s|ed|ing)?(?:\s+for)?|mak(?:e|es|ing) possible|"
            r"made possible|(?:(?:to|can|could|may|might|must|should|will|would|shall)\s+help(?:\s+to|\s+with|\s+in)?|helps(?:\s+to|\s+with|\s+in)?|helped(?:\s+to|\s+with|\s+in)?|helping(?:\s+to|\s+with|\s+in)?)|"
            r"foster(?:s|ed|ing)?|encourag(?:e|es|ed|ing)|"
            r"promot(?:e|es|ed|ing)|empower(?:s|ed|ing)?|"
            r"assist(?:s|ed|ing)?(?:\s+with|\s+in)?"
        ),
        "enables_or_supports",
    ),
    (
        cause_first_pattern(
            r"requir(?:e|es|ed|ing)|depend(?:s|ed|ing)? on|"
            r"rely|relies|relied|relying on|need(?:s|ed|ing)?|"
            r"necessitat(?:e|es|ed|ing)|presuppos(?:e|es|ed|ing)|"
            r"call(?:s|ed|ing)? for|is contingent on|is conditional on"
        ),
        "requires_or_depends_on",
    ),
    (
        cause_first_pattern(
            r"creat(?:e|es|ed|ing) a risk of|"
            r"increas(?:e|es|ed|ing) the risk of|"
            r"rais(?:e|es|ed|ing) the risk of|"
            r"pos(?:e|es|ed|ing) a risk to|risk(?:s|ed|ing)|"
            r"threaten(?:s|ed|ing)?|undermin(?:e|es|ed|ing)|"
            r"harm(?:s|ed|ing)?|jeopardi(?:s|z)(?:e|es|ed|ing)|"
            r"endanger(?:s|ed|ing)?|expos(?:e|es|ed|ing) .+? to|"
            r"compromis(?:e|es|ed|ing)|weaken(?:s|ed|ing)?"
        ),
        "risks_or_threatens",
    ),
    (
        cause_first_pattern(
            r"is expected to improve|is intended to improve|"
            r"(?:(?:to|can|could|may|might|must|should|will|would|shall)\s+improve|improves|improved|improving)|(?:(?:to|can|could|may|might|must|should|will|would|shall)\s+enhance|enhances|enhanced|enhancing)|"
            r"strengthen(?:s|ed|ing)?|advanc(?:e|es|ed|ing)|"
            r"boost(?:s|ed|ing)?|optimi(?:s|z)(?:e|es|ed|ing)|"
            r"rais(?:e|es|ed|ing) the quality of"
        ),
        "expected_to_improve",
    ),
    (
        cause_first_pattern(
            r"peut conduire à|pourrait conduire à|est susceptible de conduire à|"
            r"conduit à|conduit au|conduit aux|donne lieu à|"
            r"entraîn(?:e|es|é|ée|és|ées|ant)|caus(?:e|es|é|ée|és|ées|ant)|"
            r"provoqu(?:e|es|é|ée|és|ées|ant)|"
            r"engendr(?:e|es|é|ée|és|ées|ant)|contribu(?:e|es|é|ée|és|ées|ant) à|"
            r"génèr(?:e|es|é|ée|és|ées|ant)|déclench(?:e|es|é|ée|és|ées|ant)|"
            r"accentu(?:e|es|é|ée|és|ées|ant)|aggrav(?:e|es|é|ée|és|ées|ant)|"
            r"accroît|accroissent|augment(?:e|es|é|ée|és|ées|ant)|"
            r"amplifi(?:e|es|é|ée|és|ées|ant)"
        ),
        "causes_or_increases",
    ),
    (
        cause_first_pattern(
            r"rédui(?:t|ts|te|tes|re|sent|sant)|"
            r"prévien(?:t|nent)|limit(?:e|es|é|ée|és|ées|ant)|"
            r"atténu(?:e|es|é|ée|és|ées|ant)|diminu(?:e|es|é|ée|és|ées|ant)|"
            r"abaiss(?:e|es|é|ée|és|ées|ant)|"
            r"minimis(?:e|es|é|ée|és|ées|ant)|évit(?:e|es|é|ée|és|ées|ant)|"
            r"frein(?:e|es|é|ée|és|ées|ant)|contraint|contraignent|"
            r"protèg(?:e|es|é|ée|és|ées|ant) contre|lutt(?:e|es|é|ée|és|ées|ant) contre"
        ),
        "reduces_or_prevents",
    ),
    (
        cause_first_pattern(
            r"rend possible|rendent possible|permet(?:s|tent)? de|"
            r"permet(?:s|tent)?|soutien(?:t|nent)|facilit(?:e|es|é|ée|és|ées|ant)|"
            r"favoris(?:e|es|é|ée|és|ées|ant)|encourag(?:e|es|é|ée|és|ées|ant)|"
            r"aid(?:e|es|é|ée|és|ées|ant) à|donne les moyens de"
        ),
        "enables_or_supports",
    ),
    (
        cause_first_pattern(
            r"nécessit(?:e|es|é|ée|és|ées|ant)|dépend(?:s|ent)? de|"
            r"repos(?:e|es|é|ée|és|ées|ant) sur|exig(?:e|es|é|ée|és|ées|ant)|"
            r"requiert|requièrent|suppos(?:e|es|é|ée|és|ées|ant)|"
            r"appell(?:e|es|é|ée|és|ées|ant)|est tributaire de|"
            r"est conditionné par|est conditionnée par"
        ),
        "requires_or_depends_on",
    ),
    (
        cause_first_pattern(
            r"crée un risque de|créent un risque de|accroît le risque de|"
            r"augmente le risque de|présente un risque pour|"
            r"menac(?:e|es|é|ée|és|ées|ant)|risque de|"
            r"compromet|compromettent|nui(?:t|sent) à|met en danger|"
            r"mettent en danger|fragilis(?:e|es|é|ée|és|ées|ant)|"
            r"expos(?:e|es|é|ée|és|ées|ant) .+? à"
        ),
        "risks_or_threatens",
    ),
    (
        cause_first_pattern(
            r"devrait améliorer|vise à améliorer|"
            r"amélior(?:e|es|é|ée|és|ées|ant)|"
            r"renforc(?:e|es|é|ée|és|ées|ant)|"
            r"optimis(?:e|es|é|ée|és|ées|ant)|accroît la qualité de"
        ),
        "expected_to_improve",
    ),
]

REVERSED_PATTERNS = [
    (
        reversed_pattern(
            r"because of|because|due to|owing to|as a result of|"
            r"resulting from|caused by|driven by|triggered by|produced by|"
            r"generated by|enabled by|facilitated by|made possible by"
        ),
        "causes_or_increases",
    ),
    (
        reversed_pattern(
            r"depends? on|relies? on|is contingent on|is conditional on"
        ),
        "requires_or_depends_on",
    ),
    (
        reversed_pattern(
            r"parce que|en raison de|à cause de|du fait de|résultant de|"
            r"causé par|causée par|provoqué par|provoquée par|"
            r"entraîné par|entraînée par|généré par|générée par|"
            r"rendu possible par|rendue possible par"
        ),
        "causes_or_increases",
    ),
    (
        reversed_pattern(
            r"dépend de|repose sur|est tributaire de|"
            r"est conditionné par|est conditionnée par"
        ),
        "requires_or_depends_on",
    ),
]

PURPOSE_PATTERNS = [
    (
        cause_first_pattern(
            r"in order to|so as to|with the aim of|for the purpose of|"
            r"with a view to|designed to|intended to|"
            r"to help|to enable|to ensure|to prevent|to reduce"
        ),
        "enables_or_supports",
    ),
    (
        cause_first_pattern(
            r"afin de|dans le but de|en vue de|de manière à|de façon à|"
            r"destiné à|destinée à|conçu pour|conçue pour|visant à|"
            r"pour permettre de|pour aider à|pour assurer|pour éviter|"
            r"pour réduire"
        ),
        "enables_or_supports",
    ),
]


def split_sentences(text: str) -> list[str]:
    cleaned = html.unescape(str(text))
    cleaned = re.sub(r"\s+[•●▪◦]\s+", ". ", cleaned)
    cleaned = re.sub(
        r"\s+-\s+(?=[A-ZÀ-ÖØ-Ý][A-Za-zÀ-ÖØ-öø-ÿ ]{2,}:)",
        ". ",
        cleaned,
    )
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    if not cleaned:
        return []
    return [
        sentence.strip(" -•\t")
        for sentence in re.split(
            r"(?<=[.!?])\s+(?=[A-ZÀ-ÖØ-Ý0-9])",
            cleaned,
        )
        if sentence.strip()
    ]


def clean_side(value: str) -> str:
    cleaned = normalize_for_noise_check(value)
    cleaned = re.sub(
        r"^(?:chapter|section|article|figure|table|appendix|annex|"
        r"chapitre|partie|article|figure|tableau|annexe)\s+[A-Za-z0-9.:-]+\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(
        r"^(?:and|or|et|ou)\s+",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(
        r"\s+(?:to|de|à|pour)$",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(r"\s+", " ", cleaned).strip(" ,;:-")
    return cleaned[:500]


def is_low_information_side(value: str) -> bool:
    """Reject sides dominated by source artifacts or fragmentary tokens."""
    tokens = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9]+", str(value).lower())
    if len(tokens) < MIN_SIDE_WORDS:
        return True
    if len(tokens) > MAX_SIDE_WORDS:
        return True
    if content_token_ratio(value) < 0.45:
        return True
    artifact_hits = sum(token in CAUSAL_ARTIFACT_TOKENS for token in tokens)
    return artifact_hits >= 3


AMBIGUOUS_BASE_CUES = {
    "need", "risk", "limit", "increase", "decrease",
    "allow", "promote", "facilitate",
    "strengthen", "reduce", "prevent", "require",
}

NOUN_PREDECESSORS = {
    "a", "an", "the", "this", "that", "these", "those", "of", "for",
    "with", "without", "under", "over", "additional", "administrative",
    "continued", "digital", "educational", "existing", "financial",
    "human", "institutional", "ongoing", "practical", "professional",
    "social", "technical", "alongside", "through", "via",
    "including", "concerning", "regarding", "around",
}

SUPPORT_NOUN_FOLLOWERS = {
    "staff", "worker", "workers", "service", "services", "team", "teams",
    "function", "functions", "material", "materials", "resource",
    "resources", "network", "networks", "group", "groups", "structure",
    "structures", "mechanism", "mechanisms", "work", "personnel",
}


def is_plausible_cue_usage(sentence: str, match: re.Match) -> bool:
    """Reject common noun/adjective readings of otherwise causal cue words."""
    cue = re.sub(r"\\s+", " ", match.group("cue").lower()).strip()
    cue_start, cue_end = match.span("cue")
    before_tokens = re.findall(
        r"[A-Za-zÀ-ÖØ-öø-ÿ]+",
        sentence[:cue_start].lower(),
    )
    after_tokens = re.findall(
        r"[A-Za-zÀ-ÖØ-öø-ÿ]+",
        sentence[cue_end:].lower(),
    )
    previous_word = before_tokens[-1] if before_tokens else ""
    next_word = after_tokens[0] if after_tokens else ""

    if cue in AMBIGUOUS_BASE_CUES and previous_word in NOUN_PREDECESSORS:
        return False

    if cue == "support":
        if previous_word in NOUN_PREDECESSORS:
            return False
        if previous_word in {
            "provide", "provides", "provided", "providing",
            "obtain", "obtains", "obtained", "obtaining",
            "receive", "receives", "received", "receiving",
            "funding", "scientific",
        }:
            return False
        if next_word in SUPPORT_NOUN_FOLLOWERS | {
            "for", "of", "from", "to", "and", "or",
        }:
            return False

    if cue == "help":
        if previous_word in NOUN_PREDECESSORS:
            return False
        if next_word in {"from", "of", "for", "and", "or"}:
            return False

    if cue == "need":
        if previous_word in {"the", "a", "an", "this", "that", "identified"}:
            return False
        if next_word in {"for", "of"}:
            return False

    if cue == "risk":
        return False

    if cue in {"increase", "decrease", "limit"}:
        if previous_word in {"the", "a", "an", "this", "that", "percentage"}:
            return False

    if cue == "risks":
        if previous_word in {
            "the", "a", "an", "other", "potential", "greatest",
            "significant", "various", "several", "these", "those",
            "related", "associated",
        }:
            return False
        if next_word in {
            "is", "are", "was", "were", "that", "to", "across",
            "related", "associated", "identified", "include", "including",
        }:
            return False

    if cue in {
        "improved", "enhanced", "strengthened", "increased",
        "reduced", "limited", "generated", "created",
    }:
        if previous_word in {
            "a", "an", "the", "and", "or", "new", "better", "more",
            "less", "with", "for", "of", "is", "are", "was", "were",
            "be", "been", "being", "remains", "remained",
        }:
            return False

    if cue.endswith("ing") and previous_word in {
        "in", "for", "of", "by", "through", "while",
        "after", "before", "during",
    }:
        return False

    if cue == "need" and next_word in {
        "paper", "report", "document", "study", "article",
    }:
        return False

    if cue == "required" and (
        next_word == "by"
        or previous_word in {"as", "is", "are", "was", "were"}
    ):
        return False

    # A cue followed only by punctuation, a conjunction, or a stranded
    # preposition does not yield a meaningful effect phrase.
    if next_word in {
        "and", "or", "et", "ou", "for", "of", "from", "with",
    }:
        return False

    return True


def extract_claims(frame: pd.DataFrame, corpus_type: str) -> pd.DataFrame:
    rows = []
    counter = 0
    for _, document in frame.iterrows():
        for sentence_index, sentence in enumerate(split_sentences(document["chunk_text"])):
            lowered = sentence.lower()
            words = sentence.split()
            if (
                len(words) > MAX_SENTENCE_WORDS
                or is_source_residue(sentence)
            ):
                continue
            result = None
            for pattern, relation_type in [*REVERSED_PATTERNS, *PURPOSE_PATTERNS, *CAUSE_FIRST_PATTERNS]:
                match = re.match(pattern, sentence, flags=re.IGNORECASE)
                if not match:
                    continue
                if not is_plausible_cue_usage(sentence, match):
                    continue
                cause = clean_side(match.group("cause"))
                effect = clean_side(match.group("effect"))
                if not is_low_information_side(cause) and not is_low_information_side(effect):
                    result = (cause, relation_type, effect, match.group("cue"))
                    break
            if result is None:
                continue
            cause, relation_type, effect, cue = result
            if (
                contains_contact_or_link(cause)
                or contains_contact_or_link(effect)
                or "|" in sentence
                or "|" in cause
                or "|" in effect
            ):
                continue

            claim_signature = re.sub(
                r"[^a-zà-öø-ÿ0-9]+",
                " ",
                f"{cause} {relation_type} {effect}".lower(),
            ).strip()

            cleaned_sentence = normalize_for_noise_check(sentence)
            rows.append({
                "claim_id": f"{corpus_type}_claim_{counter:05d}",
                "corpus_type": corpus_type,
                "doc_id": document["doc_id"],
                "chunk_id": document["chunk_id"],
                "sentence_id": document["sentence_id"],
                "chunk_index": int(document["chunk_index"]),
                "filename": document["filename"],
                "source_file": document["source_file"],
                "country": document["analysis_country"],
                "heading_context": document.get("heading_context", ""),
                "source_text": cleaned_sentence,
                "context_window": document.get("context_window", ""),
                "cause": cause,
                "relation_type": relation_type,
                "relation_cue": cue,
                "effect": effect,
                "structured_claim": (
                    f"{cause} [{relation_type}] {effect}"
                ),
                "canonical_claim": cleaned_sentence,
                "nmf_topic": int(document["topic"]),
                "topic_confidence": float(document.get("topic_confidence", np.nan)),
                "extraction_method": "sentence_level_first_valid_pattern",
                "review_status": "",
                "review_notes": "",
                "claim_signature": claim_signature,
            })
            counter += 1
    result = pd.DataFrame(rows)
    if result.empty:
        return result
    # Adjacent chunks may repeat the same sentence. Keep one occurrence per
    # document while preserving identical claims from genuinely different sources.
    result = result.drop_duplicates(
        subset=["corpus_type", "doc_id", "claim_signature"]
    ).reset_index(drop=True)
    return result.drop(columns=["claim_signature"])


policy_claims = extract_claims(policy_chunks, "policy")
sentiment_claims = extract_claims(sentiment_chunks, "sentiment")
synthetic_claims = extract_claims(synthetic_chunks, "synthetic_sentiment")
if policy_claims.empty or sentiment_claims.empty:
    raise ValueError("No empirical claims were extracted.")

pd.concat([policy_claims, sentiment_claims], ignore_index=True).to_csv(
    OUTPUT_DIR / "causal_claim_extraction_audit.csv", index=False
)
print("Policy claims:", len(policy_claims))
print("Sentiment claims:", len(sentiment_claims))
print("Synthetic claims:", len(synthetic_claims))
print(pd.concat([policy_claims, sentiment_claims])["relation_type"].value_counts().to_string())


Policy claims: 3930
Sentiment claims: 761
Synthetic claims: 761
relation_type
enables_or_supports       2069
causes_or_increases       1279
requires_or_depends_on     688
expected_to_improve        291
reduces_or_prevents        265
risks_or_threatens          99


In [4]:
# ==========================================
# Step 3: Preserve the two native frozen NMF topic spaces
#
# Purpose:
# Keep policy topics as P0--P8 and sentiment topics as S0--S8. This analysis does
# not force sentiment topics into policy topics. A reviewed soft alignment, when
# needed, belongs in the independent comparison stage rather than inside the
# semantic coverage measurement itself.
# ==========================================

policy_claims["native_topic"] = policy_claims["nmf_topic"].astype(int)
policy_claims["topic_space"] = "policy"
policy_claims["topic_code"] = policy_claims["native_topic"].map(
    lambda value: f"P{int(value)}"
)

sentiment_claims["native_topic"] = sentiment_claims["nmf_topic"].astype(int)
sentiment_claims["topic_space"] = "sentiment"
sentiment_claims["topic_code"] = sentiment_claims["native_topic"].map(
    lambda value: f"S{int(value)}"
)

if not synthetic_claims.empty:
    synthetic_claims["native_topic"] = synthetic_claims["nmf_topic"].astype(int)
    synthetic_claims["topic_space"] = "sentiment"
    synthetic_claims["topic_code"] = synthetic_claims["native_topic"].map(
        lambda value: f"S{int(value)}"
    )

policy_topics["topic_space"] = "policy"
policy_topics["topic_code"] = policy_topics["topic"].astype(int).map(
    lambda value: f"P{int(value)}"
)
policy_topics["topic_description"] = build_topic_description(policy_topics)

sentiment_topics["topic_space"] = "sentiment"
sentiment_topics["topic_code"] = sentiment_topics["topic"].astype(int).map(
    lambda value: f"S{int(value)}"
)
sentiment_topics["topic_description"] = build_topic_description(sentiment_topics)

topic_descriptions = pd.concat(
    [
        policy_topics[
            [
                "topic_space", "topic_code", "topic", "topic_label",
                "topic_prototype", "keywords", "topic_description",
            ]
        ],
        sentiment_topics[
            [
                "topic_space", "topic_code", "topic", "topic_label",
                "topic_prototype", "keywords", "topic_description",
            ]
        ],
    ],
    ignore_index=True,
)
topic_descriptions.to_csv(
    OUTPUT_DIR / "native_topic_descriptions_for_comparison.csv",
    index=False,
)

policy_claims.to_csv(
    OUTPUT_DIR / "policy_semantic_claims.csv",
    index=False,
)
sentiment_claims.to_csv(
    OUTPUT_DIR / "sentiment_semantic_claims.csv",
    index=False,
)
synthetic_claims.to_csv(
    OUTPUT_DIR / "synthetic_semantic_claims.csv",
    index=False,
)

print(
    topic_descriptions[
        ["topic_space", "topic_code", "topic_label"]
    ].to_string(index=False)
)


topic_space topic_code                                                       topic_label
     policy         P0                         AI-Assisted Learning Assessment Protocols
     policy         P1                                    AI Child Protection and Rights
     policy         P2                        AI Usage and Pedagogy in School Governance
     policy         P3                AI Competency Framework and Curriculum Development
     policy         P4                     Cognitive Offloading in AI-Assisted Education
     policy         P5               Digital Learning Frameworks and Assessment Policies
     policy         P6                  AI Ethical Guidelines and School Data Protection
     policy         P7                 AI Educator Guidance and Professional Development
     policy         P8                    AI Curriculum Development and School Oversight
  sentiment         S0              Young People's AI Impact on Education and Well-being
  sentiment         S

In [5]:
# ==========================================
# Step 4: Calculate global semantic coverage and balanced directional sensitivity
#
# Purpose:
# Measure bidirectional nearest-neighbour coverage while preserving native topic
# IDs. The full-corpus results are supplemented by equal-size resampling so that
# the larger policy corpus does not obtain an automatic matching advantage.
#
# Threshold policy:
# A manually reviewed calibration file is used when available. Otherwise, the
# pooled empirical 10th percentile is retained only as a provisional lower-tail
# diagnostic. Continuous similarity and deficit values remain the main results.
# ==========================================

TOP_K = 3
BALANCED_RESAMPLES = 50
LOW_COVERAGE_QUANTILE = 0.10
THRESHOLD_AUDIT_PER_BAND = 20
MANUAL_THRESHOLD_FILE = OUTPUT_DIR / "semantic_match_threshold_manual.csv"

all_texts = (
    policy_claims["canonical_claim"].tolist()
    + sentiment_claims["canonical_claim"].tolist()
)
all_embeddings, embedding_backend = fit_text_embeddings(all_texts)
n_policy = len(policy_claims)
policy_embeddings = all_embeddings[:n_policy]
sentiment_embeddings = all_embeddings[n_policy:]


def normalized_similarity_matrix(left, right) -> np.ndarray:
    """Multiply already-normalized embeddings without a second normalization pass."""
    product = left @ right.T
    if hasattr(product, "toarray"):
        product = product.toarray()
    return np.asarray(product, dtype=np.float32)


def build_matches(
    source: pd.DataFrame,
    target: pd.DataFrame,
    source_embeddings: np.ndarray,
    target_embeddings: np.ndarray,
    direction: str,
) -> pd.DataFrame:
    """Return traceable nearest-neighbour matches without applying a threshold."""
    if len(source) == 0 or len(target) == 0:
        return pd.DataFrame()

    matrix = normalized_similarity_matrix(source_embeddings, target_embeddings)
    k = min(TOP_K, matrix.shape[1])
    order = np.argsort(matrix, axis=1)[:, ::-1][:, :k]
    scores = np.take_along_axis(matrix, order, axis=1)

    rows = []
    source_reset = source.reset_index(drop=True)
    target_reset = target.reset_index(drop=True)
    for index, source_row in source_reset.iterrows():
        best = target_reset.iloc[int(order[index, 0])]
        mean_similarity = float(scores[index].mean())
        rows.append({
            "direction": direction,
            "source_claim_id": source_row["claim_id"],
            "source_chunk_id": source_row["chunk_id"],
            "source_doc_id": source_row["doc_id"],
            "source_country": source_row["country"],
            "source_topic_space": source_row["topic_space"],
            "source_topic": int(source_row["native_topic"]),
            "source_topic_code": source_row["topic_code"],
            "source_relation_type": source_row["relation_type"],
            "source_claim": source_row["canonical_claim"],
            "source_excerpt": source_row["source_text"],
            "matched_claim_id": best["claim_id"],
            "matched_chunk_id": best["chunk_id"],
            "matched_topic_space": best["topic_space"],
            "matched_topic": int(best["native_topic"]),
            "matched_topic_code": best["topic_code"],
            "matched_claim": best["canonical_claim"],
            "matched_excerpt": best["source_text"],
            "top1_similarity": float(scores[index, 0]),
            "topk_mean_similarity": mean_similarity,
            "coverage_deficit": float(1.0 - mean_similarity),
            "embedding_method": embedding_backend["method"],
        })
    return pd.DataFrame(rows)


sentiment_to_policy = build_matches(
    sentiment_claims,
    policy_claims,
    sentiment_embeddings,
    policy_embeddings,
    "sentiment_to_policy",
)
policy_to_sentiment = build_matches(
    policy_claims,
    sentiment_claims,
    policy_embeddings,
    sentiment_embeddings,
    "policy_to_sentiment",
)
global_matches = pd.concat(
    [sentiment_to_policy, policy_to_sentiment],
    ignore_index=True,
)


def parse_manual_label(value) -> bool:
    """Convert common manual-review values to a Boolean match label."""
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in {"1", "true", "yes", "match", "matched"}:
        return True
    if text in {"0", "false", "no", "nonmatch", "non-match", "unmatched"}:
        return False
    raise ValueError(f"Unrecognised manual semantic-match label: {value!r}")


def calibrate_threshold(
    matches: pd.DataFrame,
    manual_path: Path,
) -> tuple[float, str]:
    """Use reviewed pairs when available; otherwise use a lower-tail diagnostic."""
    if manual_path.exists():
        reviewed = pd.read_csv(manual_path)
        score_column = (
            "topk_mean_similarity"
            if "topk_mean_similarity" in reviewed.columns
            else "similarity"
        )
        label_column = (
            "is_semantic_match"
            if "is_semantic_match" in reviewed.columns
            else "review_is_match"
        )
        required = {score_column, label_column}
        if not required.issubset(reviewed.columns):
            raise ValueError(
                "Manual threshold file must contain a similarity score and "
                "is_semantic_match/review_is_match."
            )

        scores = pd.to_numeric(
            reviewed[score_column],
            errors="coerce",
        )
        labels = reviewed[label_column].map(parse_manual_label)
        valid = scores.notna()
        scores = scores[valid].to_numpy(dtype=float)
        labels = labels[valid].to_numpy(dtype=bool)
        if len(scores) < 10 or labels.sum() == 0 or (~labels).sum() == 0:
            raise ValueError(
                "Manual threshold calibration needs at least 10 reviewed pairs "
                "containing both matches and non-matches."
            )

        candidates = np.unique(scores)
        best = None
        for threshold in candidates:
            predicted = scores >= threshold
            tp = int(np.sum(predicted & labels))
            fp = int(np.sum(predicted & ~labels))
            fn = int(np.sum(~predicted & labels))
            precision = tp / (tp + fp) if tp + fp else 0.0
            recall = tp / (tp + fn) if tp + fn else 0.0
            f1 = (
                2 * precision * recall / (precision + recall)
                if precision + recall
                else 0.0
            )
            candidate = (f1, precision, recall, float(threshold))
            if best is None or candidate > best:
                best = candidate

        return best[3], "manual_review_f1_calibration"

    threshold = float(
        matches["topk_mean_similarity"].quantile(LOW_COVERAGE_QUANTILE)
    )
    return threshold, (
        f"pooled_empirical_{int(LOW_COVERAGE_QUANTILE * 100)}th_"
        "percentile_provisional"
    )


coverage_threshold, threshold_method = calibrate_threshold(
    global_matches,
    MANUAL_THRESHOLD_FILE,
)
global_matches["below_calibrated_threshold"] = (
    global_matches["topk_mean_similarity"] < coverage_threshold
)
global_matches["coverage_threshold"] = coverage_threshold
global_matches["threshold_method"] = threshold_method
global_matches.to_csv(
    OUTPUT_DIR / "global_semantic_claim_matches.csv",
    index=False,
)

# Export a stratified review sheet for later threshold calibration.
audit_source = global_matches.copy()
audit_source["similarity_band"] = pd.qcut(
    audit_source["topk_mean_similarity"],
    q=[0.0, 0.10, 0.50, 0.90, 1.0],
    labels=["lowest_10pct", "lower_middle", "upper_middle", "highest_10pct"],
    duplicates="drop",
)
audit_parts = []
for _, group in audit_source.groupby(
    ["direction", "similarity_band"],
    observed=True,
):
    audit_parts.append(
        group.sample(
            n=min(THRESHOLD_AUDIT_PER_BAND, len(group)),
            random_state=RANDOM_STATE,
        )
    )
threshold_audit = pd.concat(audit_parts, ignore_index=True)
threshold_audit["review_is_match"] = ""
threshold_audit["review_notes"] = ""
threshold_audit.to_csv(
    OUTPUT_DIR / "semantic_match_threshold_audit.csv",
    index=False,
)

global_summary = global_matches.groupby(
    "direction",
    as_index=False,
).agg(
    claims=("source_claim_id", "count"),
    mean_top1_similarity=("top1_similarity", "mean"),
    mean_topk_similarity=("topk_mean_similarity", "mean"),
    mean_coverage_deficit=("coverage_deficit", "mean"),
    low_coverage_share=("below_calibrated_threshold", "mean"),
)
global_summary["coverage_threshold"] = coverage_threshold
global_summary["threshold_method"] = threshold_method
global_summary.to_csv(
    OUTPUT_DIR / "global_semantic_coverage_summary.csv",
    index=False,
)

coverage_by_topic = global_matches.groupby(
    [
        "direction",
        "source_topic_space",
        "source_topic",
        "source_topic_code",
    ],
    as_index=False,
).agg(
    claims=("source_claim_id", "count"),
    mean_topk_similarity=("topk_mean_similarity", "mean"),
    mean_coverage_deficit=("coverage_deficit", "mean"),
    low_coverage_share=("below_calibrated_threshold", "mean"),
    unique_documents=("source_doc_id", "nunique"),
)
coverage_by_topic.to_csv(
    OUTPUT_DIR / "global_semantic_coverage_by_native_topic.csv",
    index=False,
)
# Backward-compatible filename, now explicitly containing native P and S spaces.
coverage_by_topic.to_csv(
    OUTPUT_DIR / "global_semantic_coverage_by_topic.csv",
    index=False,
)

# Equal-size resampling controls for the unequal policy and sentiment claim pools.
policy_sentiment_similarity = normalized_similarity_matrix(
    policy_embeddings,
    sentiment_embeddings,
)
balanced_sample_size = min(len(policy_claims), len(sentiment_claims))
rng = np.random.default_rng(RANDOM_STATE)
balanced_global_rows = []
balanced_topic_rows = []


def topk_mean(matrix: np.ndarray, axis: int) -> np.ndarray:
    k = min(TOP_K, matrix.shape[axis])
    if axis == 1:
        selected = np.partition(
            matrix,
            matrix.shape[1] - k,
            axis=1,
        )[:, -k:]
        return selected.mean(axis=1)
    selected = np.partition(
        matrix,
        matrix.shape[0] - k,
        axis=0,
    )[-k:, :]
    return selected.mean(axis=0)


for iteration in range(BALANCED_RESAMPLES):
    policy_index = np.sort(
        rng.choice(
            len(policy_claims),
            size=balanced_sample_size,
            replace=False,
        )
    )
    sentiment_index = np.sort(
        rng.choice(
            len(sentiment_claims),
            size=balanced_sample_size,
            replace=False,
        )
    )
    sampled = policy_sentiment_similarity[
        np.ix_(policy_index, sentiment_index)
    ]

    specifications = [
        (
            "policy_to_sentiment",
            policy_claims.iloc[policy_index],
            topk_mean(sampled, axis=1),
        ),
        (
            "sentiment_to_policy",
            sentiment_claims.iloc[sentiment_index],
            topk_mean(sampled, axis=0),
        ),
    ]
    for direction, source_rows, similarities in specifications:
        deficits = 1.0 - similarities
        low_coverage = similarities < coverage_threshold
        balanced_global_rows.append({
            "iteration": iteration,
            "direction": direction,
            "mean_topk_similarity": float(similarities.mean()),
            "mean_coverage_deficit": float(deficits.mean()),
            "low_coverage_share": float(low_coverage.mean()),
        })

        iteration_frame = pd.DataFrame({
            "source_topic_space": source_rows["topic_space"].to_numpy(),
            "source_topic": source_rows["native_topic"].to_numpy(dtype=int),
            "source_topic_code": source_rows["topic_code"].to_numpy(),
            "coverage_deficit": deficits,
            "below_threshold": low_coverage,
        })
        topic_summary = iteration_frame.groupby(
            [
                "source_topic_space",
                "source_topic",
                "source_topic_code",
            ],
            as_index=False,
        ).agg(
            mean_coverage_deficit=("coverage_deficit", "mean"),
            low_coverage_share=("below_threshold", "mean"),
            claims=("source_topic_code", "size"),
        )
        topic_summary["direction"] = direction
        topic_summary["iteration"] = iteration
        balanced_topic_rows.append(topic_summary)

balanced_global_iterations = pd.DataFrame(balanced_global_rows)
balanced_topic_iterations = pd.concat(
    balanced_topic_rows,
    ignore_index=True,
)
balanced_global_iterations.to_csv(
    OUTPUT_DIR / "balanced_semantic_global_iterations.csv",
    index=False,
)
balanced_topic_iterations.to_csv(
    OUTPUT_DIR / "balanced_semantic_coverage_iterations.csv",
    index=False,
)

balanced_global_summary = balanced_global_iterations.groupby(
    "direction",
    as_index=False,
).agg(
    mean_topk_similarity=("mean_topk_similarity", "mean"),
    mean_coverage_deficit=("mean_coverage_deficit", "mean"),
    deficit_std=("mean_coverage_deficit", "std"),
    deficit_ci_low=(
        "mean_coverage_deficit",
        lambda values: float(np.quantile(values, 0.025)),
    ),
    deficit_ci_high=(
        "mean_coverage_deficit",
        lambda values: float(np.quantile(values, 0.975)),
    ),
    low_coverage_share=("low_coverage_share", "mean"),
)
balanced_global_summary["resamples"] = BALANCED_RESAMPLES
balanced_global_summary["sample_size_per_corpus"] = balanced_sample_size
balanced_global_summary["coverage_threshold"] = coverage_threshold
balanced_global_summary["threshold_method"] = threshold_method
balanced_global_summary.to_csv(
    OUTPUT_DIR / "global_balanced_semantic_coverage_summary.csv",
    index=False,
)

balanced_by_topic = balanced_topic_iterations.groupby(
    [
        "direction",
        "source_topic_space",
        "source_topic",
        "source_topic_code",
    ],
    as_index=False,
).agg(
    mean_coverage_deficit=("mean_coverage_deficit", "mean"),
    deficit_std=("mean_coverage_deficit", "std"),
    low_coverage_share=("low_coverage_share", "mean"),
    mean_sampled_claims=("claims", "mean"),
)
balanced_by_topic.to_csv(
    OUTPUT_DIR / "global_balanced_semantic_coverage_by_native_topic.csv",
    index=False,
)

print(global_summary.to_string(index=False))
print("\nBalanced directional analysis:")
print(balanced_global_summary.to_string(index=False))
print("\nCoverage threshold:", coverage_threshold, threshold_method)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

          direction  claims  mean_top1_similarity  mean_topk_similarity  mean_coverage_deficit  low_coverage_share  coverage_threshold                             threshold_method
policy_to_sentiment    3930              0.661758              0.641370               0.358630            0.109160            0.511861 pooled_empirical_10th_percentile_provisional
sentiment_to_policy     761              0.694468              0.675473               0.324527            0.052562            0.511861 pooled_empirical_10th_percentile_provisional

Balanced directional analysis:
          direction  mean_topk_similarity  mean_coverage_deficit  deficit_std  deficit_ci_low  deficit_ci_high  low_coverage_share  resamples  sample_size_per_corpus  coverage_threshold                             threshold_method
policy_to_sentiment              0.641457               0.358543     0.002785        0.353392         0.363795            0.106675         50                     761            0.511861 pooled_empi

In [6]:
# ==========================================
# Step 5: Calculate country-level semantic coverage in native topic spaces
#
# Purpose:
# Repeat matching within each supported country. Policy-source claims remain
# grouped under P0--P8 and sentiment-source claims remain grouped under S0--S8.
# Synthetic claims never fill missing country evidence.
# ==========================================

country_rows = []
country_matches = []
all_countries = sorted(
    set(policy_claims["country"]).union(set(sentiment_claims["country"]))
)

for country in all_countries:
    policy_subset = policy_claims[
        policy_claims["country"].eq(country)
    ].copy()
    sentiment_subset = sentiment_claims[
        sentiment_claims["country"].eq(country)
    ].copy()

    sufficient = (
        len(policy_subset) >= MIN_COUNTRY_ITEMS
        and len(sentiment_subset) >= MIN_COUNTRY_ITEMS
    )
    country_rows.append({
        "country": country,
        "policy_claims": len(policy_subset),
        "sentiment_claims": len(sentiment_subset),
        "coverage_status": (
            "included" if sufficient else "insufficient evidence"
        ),
    })
    if not sufficient:
        continue

    policy_positions = policy_subset.index.to_numpy()
    sentiment_positions = sentiment_subset.index.to_numpy()

    sentiment_country_matches = build_matches(
        sentiment_subset.reset_index(drop=True),
        policy_subset.reset_index(drop=True),
        sentiment_embeddings[sentiment_positions],
        policy_embeddings[policy_positions],
        "sentiment_to_policy",
    )
    policy_country_matches = build_matches(
        policy_subset.reset_index(drop=True),
        sentiment_subset.reset_index(drop=True),
        policy_embeddings[policy_positions],
        sentiment_embeddings[sentiment_positions],
        "policy_to_sentiment",
    )

    for frame in [sentiment_country_matches, policy_country_matches]:
        frame["analysis_country"] = country
        frame["below_calibrated_threshold"] = (
            frame["topk_mean_similarity"] < coverage_threshold
        )
        frame["coverage_threshold"] = coverage_threshold
        frame["threshold_method"] = threshold_method

    country_matches.extend(
        [sentiment_country_matches, policy_country_matches]
    )

country_coverage = pd.DataFrame(country_rows)
country_match_df = (
    pd.concat(country_matches, ignore_index=True)
    if country_matches
    else pd.DataFrame()
)

country_coverage.to_csv(
    OUTPUT_DIR / "country_semantic_coverage_status.csv",
    index=False,
)
country_match_df.to_csv(
    OUTPUT_DIR / "country_semantic_claim_matches.csv",
    index=False,
)

if not country_match_df.empty:
    country_by_topic = country_match_df.groupby(
        [
            "analysis_country",
            "direction",
            "source_topic_space",
            "source_topic",
            "source_topic_code",
        ],
        as_index=False,
    ).agg(
        claims=("source_claim_id", "count"),
        mean_coverage_deficit=("coverage_deficit", "mean"),
        mean_topk_similarity=("topk_mean_similarity", "mean"),
        low_coverage_share=("below_calibrated_threshold", "mean"),
        unique_documents=("source_doc_id", "nunique"),
    )
else:
    country_by_topic = pd.DataFrame()

country_by_topic.to_csv(
    OUTPUT_DIR / "country_semantic_coverage_by_native_topic.csv",
    index=False,
)
country_by_topic.to_csv(
    OUTPUT_DIR / "country_semantic_coverage_by_topic.csv",
    index=False,
)
print(country_coverage.to_string(index=False))


  country  policy_claims  sentiment_claims       coverage_status
australia            787                 0 insufficient evidence
   france            378                34              included
  ireland            988               170              included
    other            538               557              included
      usa           1239                 0 insufficient evidence


In [7]:
# ==========================================
# Step 6: Mine original-document evidence for the strongest coverage gaps
#
# Purpose:
# Ground low-coverage findings in original excerpts, headings, neighbouring text,
# and source diversity. Native topic IDs are retained so documentary evidence is
# not distorted by an automatic cross-corpus topic mapping.
# ==========================================

ranked_evidence = global_matches.sort_values(
    ["coverage_deficit", "source_topic_space", "source_topic"],
    ascending=[False, True, True],
).copy()
ranked_evidence["within_topic_rank"] = ranked_evidence.groupby(
    ["direction", "source_topic_space", "source_topic_code"]
)["coverage_deficit"].rank(
    ascending=False,
    method="first",
)
global_evidence = ranked_evidence[
    ranked_evidence["within_topic_rank"] <= 3
].copy()
global_evidence.to_csv(
    OUTPUT_DIR / "global_semantic_gap_evidence.csv",
    index=False,
)

source_support = global_matches.groupby(
    [
        "direction",
        "source_topic_space",
        "source_topic",
        "source_topic_code",
    ],
    as_index=False,
).agg(
    claims=("source_claim_id", "count"),
    unique_documents=("source_doc_id", "nunique"),
    mean_coverage_deficit=("coverage_deficit", "mean"),
)
source_support.to_csv(
    OUTPUT_DIR / "global_semantic_source_support.csv",
    index=False,
)

print(
    global_evidence[
        [
            "direction",
            "source_topic_code",
            "coverage_deficit",
            "source_excerpt",
        ]
    ].head(12).to_string(index=False)
)

# Visual-only native-topic connections
#
# A policy and sentiment topic are connected only when each is the other's
# strongest destination in the claim-level top-1 matching graph, the symmetric
# connection score is sufficiently large, and both directions have documentary
# support. This does not remap topics or alter coverage statistics.

TOPIC_CONNECTION_MIN_SCORE = 0.12
TOPIC_CONNECTION_MIN_FORWARD_LINKS = 3
TOPIC_CONNECTION_MIN_REVERSE_LINKS = 2

policy_to_sentiment_links = global_matches[
    global_matches["direction"].eq("policy_to_sentiment")
].copy()
sentiment_to_policy_links = global_matches[
    global_matches["direction"].eq("sentiment_to_policy")
].copy()

policy_topic_ids = sorted(
    policy_to_sentiment_links["source_topic"].astype(int).unique()
)
sentiment_topic_ids = sorted(
    sentiment_to_policy_links["source_topic"].astype(int).unique()
)

forward_weight = (
    policy_to_sentiment_links.groupby(
        ["source_topic", "matched_topic"]
    )["top1_similarity"]
    .sum()
    .unstack(fill_value=0.0)
    .reindex(
        index=policy_topic_ids,
        columns=sentiment_topic_ids,
        fill_value=0.0,
    )
)
forward_count = (
    policy_to_sentiment_links.groupby(
        ["source_topic", "matched_topic"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        index=policy_topic_ids,
        columns=sentiment_topic_ids,
        fill_value=0,
    )
)
reverse_weight = (
    sentiment_to_policy_links.groupby(
        ["matched_topic", "source_topic"]
    )["top1_similarity"]
    .sum()
    .unstack(fill_value=0.0)
    .reindex(
        index=policy_topic_ids,
        columns=sentiment_topic_ids,
        fill_value=0.0,
    )
)
reverse_count = (
    sentiment_to_policy_links.groupby(
        ["matched_topic", "source_topic"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        index=policy_topic_ids,
        columns=sentiment_topic_ids,
        fill_value=0,
    )
)

forward_share = forward_weight.div(
    forward_weight.sum(axis=1).replace(0.0, np.nan),
    axis=0,
).fillna(0.0)
reverse_share = reverse_weight.div(
    reverse_weight.sum(axis=0).replace(0.0, np.nan),
    axis=1,
).fillna(0.0)
connection_score = np.sqrt(forward_share * reverse_share)

connection_rows = []
for policy_topic in policy_topic_ids:
    for sentiment_topic in sentiment_topic_ids:
        score = float(connection_score.loc[policy_topic, sentiment_topic])
        policy_best = int(connection_score.loc[policy_topic].idxmax())
        sentiment_best = int(connection_score[sentiment_topic].idxmax())
        mutual_best = (
            policy_best == sentiment_topic
            and sentiment_best == policy_topic
        )
        f_count = int(forward_count.loc[policy_topic, sentiment_topic])
        r_count = int(reverse_count.loc[policy_topic, sentiment_topic])
        connected = (
            mutual_best
            and score >= TOPIC_CONNECTION_MIN_SCORE
            and f_count >= TOPIC_CONNECTION_MIN_FORWARD_LINKS
            and r_count >= TOPIC_CONNECTION_MIN_REVERSE_LINKS
        )
        connection_rows.append({
            "policy_topic": policy_topic,
            "policy_topic_code": f"P{policy_topic}",
            "sentiment_topic": sentiment_topic,
            "sentiment_topic_code": f"S{sentiment_topic}",
            "forward_weight_share": float(
                forward_share.loc[policy_topic, sentiment_topic]
            ),
            "reverse_weight_share": float(
                reverse_share.loc[policy_topic, sentiment_topic]
            ),
            "connection_score": score,
            "forward_links": f_count,
            "reverse_links": r_count,
            "mutual_best": mutual_best,
            "connected_for_visualization": connected,
        })

topic_connection_matrix = pd.DataFrame(connection_rows)
topic_connection_matrix.to_csv(
    OUTPUT_DIR / "semantic_topic_connection_matrix.csv",
    index=False,
)

topic_connections = (
    topic_connection_matrix[
        topic_connection_matrix["connected_for_visualization"]
    ]
    .sort_values("policy_topic")
    .reset_index(drop=True)
)
topic_connections.to_csv(
    OUTPUT_DIR / "semantic_topic_connections.csv",
    index=False,
)

print("High-confidence visual topic connections:")
if topic_connections.empty:
    print("None; all native topics will remain separate.")
else:
    print(
        topic_connections[
            [
                "policy_topic_code",
                "sentiment_topic_code",
                "connection_score",
                "forward_links",
                "reverse_links",
            ]
        ].to_string(index=False)
    )


          direction source_topic_code  coverage_deficit                                                                                                                                                                                                                            source_excerpt
policy_to_sentiment                P8          0.761373                                                               Fideliz Apilado, Laicia Gagnier, Samuel Grimonprez, Glen Hertelendy, Michela Pagano and Xianglei Zheng from the same Unit also supported the production of the publication.
policy_to_sentiment                P3          0.749442                                                                                                                                                                                            jumping backwards to avoid a sensed obstacle).
policy_to_sentiment                P8          0.716446 Contracts and Procurement This document is not intended to provide legal a

In [8]:
# ==========================================
# Step 7: Run synthetic-data sensitivity in the native sentiment topic space
#
# Purpose:
# Compare empirical and synthetic sentiment claims within S0--S8. Synthetic data
# are introduced only after the empirical analysis and do not replace country
# evidence.
# ==========================================

synthetic_sensitivity = pd.DataFrame()

if not synthetic_claims.empty:
    synthetic_embeddings = transform_text_embeddings(
        synthetic_claims["canonical_claim"].tolist(),
        embedding_backend,
    )
    synthetic_matches = build_matches(
        synthetic_claims,
        policy_claims,
        synthetic_embeddings,
        policy_embeddings,
        "synthetic_to_policy",
    )
    synthetic_matches["below_calibrated_threshold"] = (
        synthetic_matches["topk_mean_similarity"] < coverage_threshold
    )
    synthetic_matches["coverage_threshold"] = coverage_threshold
    synthetic_matches["threshold_method"] = threshold_method
    synthetic_matches.to_csv(
        OUTPUT_DIR / "synthetic_semantic_claim_matches.csv",
        index=False,
    )

    grouping = [
        "source_topic_space",
        "source_topic",
        "source_topic_code",
    ]
    empirical = sentiment_to_policy.groupby(
        grouping,
        as_index=False,
    ).agg(
        empirical_mean_deficit=("coverage_deficit", "mean"),
        empirical_claims=("source_claim_id", "count"),
        empirical_documents=("source_doc_id", "nunique"),
    )
    synthetic = synthetic_matches.groupby(
        grouping,
        as_index=False,
    ).agg(
        synthetic_mean_deficit=("coverage_deficit", "mean"),
        synthetic_claims=("source_claim_id", "count"),
        synthetic_documents=("source_doc_id", "nunique"),
    )

    synthetic_sensitivity = empirical.merge(
        synthetic,
        on=grouping,
        how="outer",
    )

    empirical_available = synthetic_sensitivity[
        "empirical_mean_deficit"
    ].notna()
    synthetic_available = synthetic_sensitivity[
        "synthetic_mean_deficit"
    ].notna()

    synthetic_sensitivity["comparison_status"] = np.select(
        [
            empirical_available & synthetic_available,
            empirical_available & ~synthetic_available,
            ~empirical_available & synthetic_available,
        ],
        [
            "supported",
            "insufficient_synthetic_evidence",
            "insufficient_empirical_evidence",
        ],
        default="unsupported",
    )
    synthetic_sensitivity["deficit_change"] = np.where(
        synthetic_sensitivity["comparison_status"].eq("supported"),
        synthetic_sensitivity["synthetic_mean_deficit"]
        - synthetic_sensitivity["empirical_mean_deficit"],
        np.nan,
    )
    synthetic_sensitivity.to_csv(
        OUTPUT_DIR / "synthetic_semantic_coverage_sensitivity.csv",
        index=False,
    )

print("Synthetic sensitivity rows:", len(synthetic_sensitivity))
if not synthetic_sensitivity.empty:
    print(
        synthetic_sensitivity["comparison_status"]
        .value_counts(dropna=False)
        .to_string()
    )


Synthetic sensitivity rows: 9
comparison_status
supported                          8
insufficient_synthetic_evidence    1


In [9]:
# ==========================================
# Step 8: Generate report-ready figures using native topic IDs only
#
# Purpose:
# Retain the original blue/orange visual form. Native topics are placed side by
# side only when the bidirectional claim graph supports a high-confidence
# connection. Unconnected topics remain separate.
# ==========================================

BLUE = "#1f77b4"
ORANGE = "#ff7f0e"
DIRECTION_ORDER = ["policy_to_sentiment", "sentiment_to_policy"]

# Old-style overlaid histogram: same bins, blue policy direction, orange
# sentiment direction, and transparent overlap.
policy_similarity = global_matches.loc[
    global_matches["direction"].eq("policy_to_sentiment"),
    "topk_mean_similarity",
].dropna().to_numpy()
sentiment_similarity = global_matches.loc[
    global_matches["direction"].eq("sentiment_to_policy"),
    "topk_mean_similarity",
].dropna().to_numpy()

all_similarity = np.concatenate(
    [policy_similarity, sentiment_similarity]
)
similarity_bins = np.linspace(
    float(all_similarity.min()),
    float(all_similarity.max()),
    25,
)

fig, ax = plt.subplots(figsize=(13.5, 6.3))
ax.hist(
    policy_similarity,
    bins=similarity_bins,
    color=BLUE,
    alpha=0.52,
    label="policy_to_sentiment",
)
ax.hist(
    sentiment_similarity,
    bins=similarity_bins,
    color=ORANGE,
    alpha=0.52,
    label="sentiment_to_policy",
)
ax.set_xlabel("Top-k semantic similarity")
ax.set_ylabel("Causal claims")
ax.legend(loc="upper right")
fig.tight_layout()
save_verified_png(
    fig,
    IMG_DIR / "semantic_similarity_histogram.png",
)

# Compatibility filename matching the earlier report figure.
shutil.copyfile(
    IMG_DIR / "semantic_similarity_histogram.png",
    IMG_DIR / "global_semantic_coverage_distribution.png",
)

# Balanced topic deficits are used because both directions are then estimated
# from equal-size matching pools.
policy_topic_deficit = (
    balanced_by_topic[
        balanced_by_topic["direction"].eq("policy_to_sentiment")
    ]
    .set_index("source_topic_code")["mean_coverage_deficit"]
)
sentiment_topic_deficit = (
    balanced_by_topic[
        balanced_by_topic["direction"].eq("sentiment_to_policy")
    ]
    .set_index("source_topic_code")["mean_coverage_deficit"]
)

connected_plot_rows = []
for _, connection in topic_connections.iterrows():
    policy_code = connection["policy_topic_code"]
    sentiment_code = connection["sentiment_topic_code"]
    if (
        policy_code in policy_topic_deficit.index
        and sentiment_code in sentiment_topic_deficit.index
    ):
        connected_plot_rows.append({
            "pair_code": f"{policy_code} ↔ {sentiment_code}",
            "policy_topic_code": policy_code,
            "sentiment_topic_code": sentiment_code,
            "policy_deficit": float(policy_topic_deficit.loc[policy_code]),
            "sentiment_deficit": float(sentiment_topic_deficit.loc[sentiment_code]),
            "connection_score": float(connection["connection_score"]),
        })

connected_topic_deficits = pd.DataFrame(connected_plot_rows)
connected_topic_deficits.to_csv(
    OUTPUT_DIR / "semantic_deficit_by_connected_topic.csv",
    index=False,
)

connected_policy_codes = set(
    connected_topic_deficits.get("policy_topic_code", pd.Series(dtype=str))
)
connected_sentiment_codes = set(
    connected_topic_deficits.get("sentiment_topic_code", pd.Series(dtype=str))
)

unmatched_rows = []
for topic_code, value in policy_topic_deficit.items():
    if topic_code not in connected_policy_codes:
        unmatched_rows.append({
            "topic_code": topic_code,
            "direction": "policy_to_sentiment",
            "mean_coverage_deficit": float(value),
        })
for topic_code, value in sentiment_topic_deficit.items():
    if topic_code not in connected_sentiment_codes:
        unmatched_rows.append({
            "topic_code": topic_code,
            "direction": "sentiment_to_policy",
            "mean_coverage_deficit": float(value),
        })

unmatched_topic_deficits = pd.DataFrame(unmatched_rows)
unmatched_topic_deficits.to_csv(
    OUTPUT_DIR / "semantic_deficit_by_unmatched_topic.csv",
    index=False,
)

# One old-style topic figure: connected P↔S pairs appear side by side; native
# topics without a supported connection remain as single blue or orange bars.
plot_groups = []
for _, row in connected_topic_deficits.iterrows():
    plot_groups.append({
        "label": row["pair_code"],
        "policy_value": row["policy_deficit"],
        "sentiment_value": row["sentiment_deficit"],
    })
if not unmatched_topic_deficits.empty:
    order_frame = unmatched_topic_deficits.copy()
    order_frame["topic_number"] = (
        order_frame["topic_code"].str.extract(r"(\d+)").astype(int)
    )
    order_frame = order_frame.sort_values(["direction", "topic_number"])
    for _, row in order_frame.iterrows():
        plot_groups.append({
            "label": row["topic_code"],
            "policy_value": (
                row["mean_coverage_deficit"]
                if row["direction"] == "policy_to_sentiment"
                else np.nan
            ),
            "sentiment_value": (
                row["mean_coverage_deficit"]
                if row["direction"] == "sentiment_to_policy"
                else np.nan
            ),
        })

plot_groups = pd.DataFrame(plot_groups)
x_positions = np.arange(len(plot_groups))
bar_width = 0.36
fig, ax = plt.subplots(figsize=(max(12.0, 0.85 * len(plot_groups)), 7.0))
policy_values = plot_groups["policy_value"].to_numpy(dtype=float)
sentiment_values = plot_groups["sentiment_value"].to_numpy(dtype=float)
policy_mask = np.isfinite(policy_values)
sentiment_mask = np.isfinite(sentiment_values)
ax.bar(
    x_positions[policy_mask] - np.where(
        sentiment_mask[policy_mask], bar_width / 2, 0.0
    ),
    policy_values[policy_mask],
    bar_width,
    color=BLUE,
    label="policy_to_sentiment",
)
ax.bar(
    x_positions[sentiment_mask] + np.where(
        policy_mask[sentiment_mask], bar_width / 2, 0.0
    ),
    sentiment_values[sentiment_mask],
    bar_width,
    color=ORANGE,
    label="sentiment_to_policy",
)
ax.set_xticks(x_positions)
ax.set_xticklabels(plot_groups["label"], rotation=90)
ax.set_xlabel("Connected pairs and unmatched native topic IDs")
ax.set_ylabel("Balanced mean semantic coverage deficit")
ax.legend(loc="upper right")
fig.tight_layout()
save_verified_png(
    fig,
    IMG_DIR / "global_semantic_deficit_by_topic_id.png",
)

# Histogram of the 50 equal-size resampling iterations.
balanced_groups = [
    balanced_global_iterations.loc[
        balanced_global_iterations["direction"].eq(direction),
        "mean_coverage_deficit",
    ].dropna().to_numpy()
    for direction in DIRECTION_ORDER
]
all_balanced = np.concatenate(balanced_groups)
balanced_bins = np.linspace(
    float(all_balanced.min()),
    float(all_balanced.max()),
    16,
)

fig, ax = plt.subplots(figsize=(10.5, 6.0))
ax.hist(
    balanced_groups[0],
    bins=balanced_bins,
    color=BLUE,
    alpha=0.52,
    label="policy_to_sentiment",
)
ax.hist(
    balanced_groups[1],
    bins=balanced_bins,
    color=ORANGE,
    alpha=0.52,
    label="sentiment_to_policy",
)
ax.set_xlabel("Balanced mean semantic coverage deficit")
ax.set_ylabel("Resampling iterations")
ax.legend(loc="upper right")
fig.tight_layout()
save_verified_png(
    fig,
    IMG_DIR / "balanced_semantic_deficit_histogram.png",
)

if not country_by_topic.empty:
    country_figure_specs = [
        (
            "policy_to_sentiment",
            "country_policy_to_sentiment_deficit_by_topic_id.png",
            "Native policy topic ID",
            "Blues",
        ),
        (
            "sentiment_to_policy",
            "country_sentiment_to_policy_deficit_by_topic_id.png",
            "Native sentiment topic ID",
            "Oranges",
        ),
    ]

    for direction, filename, axis_label, colour_map in country_figure_specs:
        direction_frame = country_by_topic[
            country_by_topic["direction"].eq(direction)
        ]
        if direction_frame.empty:
            continue

        matrix = direction_frame.pivot(
            index="source_topic_code",
            columns="analysis_country",
            values="mean_coverage_deficit",
        ).sort_index()

        fig, ax = plt.subplots(
            figsize=(7 + 1.3 * len(matrix.columns), 7)
        )
        image = ax.imshow(
            matrix.to_numpy(),
            aspect="auto",
            cmap=colour_map,
        )
        ax.set_xticks(np.arange(len(matrix.columns)))
        ax.set_xticklabels(matrix.columns)
        ax.set_yticks(np.arange(len(matrix.index)))
        ax.set_yticklabels(matrix.index)
        ax.set_ylabel(axis_label)
        fig.colorbar(
            image,
            ax=ax,
            label="Mean semantic coverage deficit",
        )
        fig.tight_layout()
        save_verified_png(fig, IMG_DIR / filename)

synthetic_png = IMG_DIR / "synthetic_semantic_sensitivity_by_topic_id.png"
supported_sensitivity = synthetic_sensitivity[
    synthetic_sensitivity["comparison_status"].eq("supported")
].copy() if not synthetic_sensitivity.empty else pd.DataFrame()

if not supported_sensitivity.empty:
    plot_syn = supported_sensitivity.sort_values("source_topic").copy()
    plot_syn["deficit_change"] = pd.to_numeric(
        plot_syn["deficit_change"],
        errors="coerce",
    )
    plot_syn = plot_syn.replace(
        [np.inf, -np.inf],
        np.nan,
    ).dropna(subset=["deficit_change"])

    if plot_syn.empty:
        raise ValueError(
            "Supported synthetic comparisons contain no finite deficit changes."
        )

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(
        plot_syn["source_topic_code"].astype(str),
        plot_syn["deficit_change"].astype(float),
        color=ORANGE,
    )
    ax.axhline(0.0, color="black", linewidth=1)
    ax.set_xlabel("Native sentiment topic ID")
    ax.set_ylabel("Synthetic minus empirical deficit")
    fig.tight_layout()
    save_verified_png(fig, synthetic_png)
else:
    synthetic_png.unlink(missing_ok=True)
    print(
        "Synthetic sensitivity figure was not created because no topic had "
        "both empirical and synthetic evidence."
    )


Saved verified PNG: /home/nsirim/Github/mscdsa/msc/progress/causal_nlp/img/semantic_coverage/semantic_similarity_histogram.png mode=RGB size=(1608, 743) bytes=23,504
Saved verified PNG: /home/nsirim/Github/mscdsa/msc/progress/causal_nlp/img/semantic_coverage/global_semantic_deficit_by_topic_id.png mode=RGB size=(1428, 828) bytes=35,703
Saved verified PNG: /home/nsirim/Github/mscdsa/msc/progress/causal_nlp/img/semantic_coverage/balanced_semantic_deficit_histogram.png mode=RGB size=(1248, 707) bytes=26,500
Saved verified PNG: /home/nsirim/Github/mscdsa/msc/progress/causal_nlp/img/semantic_coverage/country_policy_to_sentiment_deficit_by_topic_id.png mode=RGB size=(1226, 827) bytes=29,250
Saved verified PNG: /home/nsirim/Github/mscdsa/msc/progress/causal_nlp/img/semantic_coverage/country_sentiment_to_policy_deficit_by_topic_id.png mode=RGB size=(1226, 827) bytes=34,109
Saved verified PNG: /home/nsirim/Github/mscdsa/msc/progress/causal_nlp/img/semantic_coverage/synthetic_semantic_sensitivit

In [10]:
# ==========================================
# Step 9: Validate generated PNG files
#
# Purpose:
# Confirm that every report figure exists, is non-empty, and can be decoded
# before the notebook completes.
# ==========================================

expected_pngs = [
    IMG_DIR / "semantic_similarity_histogram.png",
    IMG_DIR / "global_semantic_coverage_distribution.png",
    IMG_DIR / "balanced_semantic_deficit_histogram.png",
]
if not connected_topic_deficits.empty:
    expected_pngs.append(
        IMG_DIR / "global_semantic_deficit_by_topic_id.png"
    )
if not country_by_topic.empty:
    if country_by_topic["direction"].eq("policy_to_sentiment").any():
        expected_pngs.append(
            IMG_DIR
            / "country_policy_to_sentiment_deficit_by_topic_id.png"
        )
    if country_by_topic["direction"].eq("sentiment_to_policy").any():
        expected_pngs.append(
            IMG_DIR
            / "country_sentiment_to_policy_deficit_by_topic_id.png"
        )
if not supported_sensitivity.empty:
    expected_pngs.append(
        IMG_DIR / "synthetic_semantic_sensitivity_by_topic_id.png"
    )

png_validation_rows = []
for png_path in expected_pngs:
    exists = png_path.exists()
    size_bytes = png_path.stat().st_size if exists else 0
    readable = False
    image_mode = ""
    image_width = 0
    image_height = 0

    if exists and size_bytes > 0:
        with Image.open(png_path) as image:
            image.verify()
        with Image.open(png_path) as image:
            image.load()
            readable = True
            image_mode = image.mode
            image_width, image_height = image.size

    png_validation_rows.append({
        "filename": png_path.name,
        "exists": exists,
        "size_bytes": size_bytes,
        "readable": readable,
        "mode": image_mode,
        "width": image_width,
        "height": image_height,
    })

png_validation = pd.DataFrame(png_validation_rows)
png_validation.to_csv(
    OUTPUT_DIR / "png_validation.csv",
    index=False,
)
if not png_validation["readable"].all():
    raise OSError(
        "One or more PNG files could not be decoded. "
        "See png_validation.csv."
    )
print(png_validation.to_string(index=False))


                                           filename  exists  size_bytes  readable mode  width  height
                  semantic_similarity_histogram.png    True       23504      True  RGB   1608     743
          global_semantic_coverage_distribution.png    True       23504      True  RGB   1608     743
            balanced_semantic_deficit_histogram.png    True       26500      True  RGB   1248     707
            global_semantic_deficit_by_topic_id.png    True       35703      True  RGB   1428     828
country_policy_to_sentiment_deficit_by_topic_id.png    True       29250      True  RGB   1226     827
country_sentiment_to_policy_deficit_by_topic_id.png    True       34109      True  RGB   1226     827
     synthetic_semantic_sensitivity_by_topic_id.png    True       22371      True  RGB   1068     587


In [11]:
# ==========================================
# Step 10: Export run summary and references
# ==========================================

run_summary = pd.DataFrame([{
    "method": "semantic_causal_claim_coverage",
    "independent_source": "shared_clean_sentence_inventory_from_chunks_all.csv",
    "shared_cleaning_version": CLEANING_VERSION,
    "shared_inventory_sha256": shared_cleaning_metadata["inventory_sha256"],
    "shared_inventory_rows": shared_cleaning_metadata["inventory_rows"],
    "policy_claims": len(policy_claims),
    "sentiment_claims": len(sentiment_claims),
    "synthetic_claims": len(synthetic_claims),
    "embedding_method": embedding_backend["method"],
    "topic_reporting": "native_policy_P0_P8_and_sentiment_S0_S8",
    "crosswalk_applied": False,
    "top_k": TOP_K,
    "coverage_threshold": coverage_threshold,
    "threshold_method": threshold_method,
    "balanced_resamples": BALANCED_RESAMPLES,
    "balanced_sample_size": balanced_sample_size,
    "reads_causal_frame_outputs": False,
}])
run_summary.to_csv(
    OUTPUT_DIR / "semantic_run_summary.csv",
    index=False,
)

pd.DataFrame([
    {
        "method": "NMF contextual measurement",
        "citation_key": "lee2001algorithms",
    },
    {
        "method": "Multilingual sentence embeddings",
        "citation_key": "reimers2019sentencebert",
    },
    {
        "method": "Balanced resampling",
        "citation_key": "efron1979bootstrap",
    },
    {
        "method": "Causal inference with text measurements",
        "citation_key": "egami2022causal",
    },
    {
        "method": "Directed content analysis",
        "citation_key": "hsieh2005three",
    },
]).to_csv(
    OUTPUT_DIR / "method_references.csv",
    index=False,
)

print(run_summary.to_string(index=False))
print("Semantic causal-claim coverage completed independently.")


                        method                                  independent_source shared_cleaning_version                                          shared_inventory_sha256  shared_inventory_rows  policy_claims  sentiment_claims  synthetic_claims                                                                 embedding_method                         topic_reporting  crosswalk_applied  top_k  coverage_threshold                             threshold_method  balanced_resamples  balanced_sample_size  reads_causal_frame_outputs
semantic_causal_claim_coverage shared_clean_sentence_inventory_from_chunks_all.csv      shared-causal-text 59f25d414f8672b5fd0423a5c35ea1452e3d3d051a4d045d2c378c142c40f96a                  21591           3930               761               761 sentence_transformer:sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 native_policy_P0_P8_and_sentiment_S0_S8              False      3            0.511861 pooled_empirical_10th_percentile_provisional               